# Credit Risk Analysis - Exploratory Data Analysis

**Dataset:** LendingClub 2015-2018 · 893,914 loans  
**Author:** Manuel  
**Stack:** Python · Pandas · Seaborn · Matplotlib


In 2025, the Central Bank of Costa Rica (BCCR) identified a paradox: loan delinquency kept rising even as the economy improved - wages went up, unemployment fell, interest rates dropped. The BCCR pointed to over-indebtedness and longer loan terms as possible explanations.

Since individual borrower data is not publicly available in Costa Rica, this analysis uses LendingClub as a real-world proxy to explore whether those same patterns show up at the borrower level.

**Three questions this notebook explores:**
1. Does loan term show a stronger association with default than income level?
2. Do borrowers who defaulted have meaningfully different income profiles?
3. Is higher DTI (over-indebtedness) associated with higher default rates?

 **Note:** Place **lending_club_clean.csv** in the same folder as this notebook before running.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load data - place lending_club_clean.csv in the same folder as this notebook
df = pd.read_csv('lending_club_clean.csv')

print(f"Rows loaded: {len(df)}")
print(f"Columns: {list(df.columns)}")

## Visualization 1 - Default Rate by Income Decile and Loan Term

This is the key finding of the project. Borrowers are split into 10 equal groups by income, and default rates are compared between 36-month and 60-month loans at each income level.

The two lines never cross - 60-month loans consistently default more than 36-month loans across every income group. The richest borrowers with a 60-month loan (26.16%) default at a higher rate than the poorest borrowers with a 36-month loan (23.94%).

In [ ]:
# Data sourced from SQL Analysis 5 (credit_risk_queries.sql)
# Query: LOAN TERM x INCOME DECILE

with_deciles = df.copy()
with_deciles['income_decile'] = pd.qcut(with_deciles['annual_inc'], q=10, labels=range(1,11))

v1_data = with_deciles.groupby(['income_decile', 'term'])['default_flag'].mean().reset_index()
v1_data['default_rate'] = (v1_data['default_flag'] * 100).round(2)
v1_data['income_decile'] = v1_data['income_decile'].astype(int)

plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(10, 6))

for term, color in [('36 months', '#4FC3F7'), ('60 months', '#EF5350')]:
    data = v1_data[v1_data['term'] == term]
    ax.plot(data['income_decile'], data['default_rate'],
            marker='o', linewidth=2.5, label=term, color=color)

ax.set_title('Default Rate by Income Decile and Loan Term', fontsize=15, pad=15)
ax.set_xlabel('Income Decile (1 = lowest, 10 = highest)', fontsize=12)
ax.set_ylabel('Default Rate (%)', fontsize=12)
ax.legend(title='Loan Term', fontsize=11)
ax.grid(True, alpha=0.2)
ax.set_xticks(range(1, 11))

plt.tight_layout()
plt.savefig('v1_term_vs_income.png', dpi=150, bbox_inches='tight')
plt.show()

## Visualization 2 - Income Distribution by Loan Outcome

If income were the main driver of default, we would expect very different distributions between borrowers who paid and those who defaulted. The KDE curves show the opposite - both groups have nearly identical income distributions, with most borrowers concentrated around $40,000–$80,000 per year.

**Note:** Filtered to incomes between $15,000 and $300,000 to remove extreme outliers. KDE normalizes density so curves are comparable in shape, not in volume.

In [ ]:
# Raw data from lending_club_clean table
# Filtered to annual_inc between $15,000 and $300,000 to remove extreme outliers

plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(10, 6))

df_filtered = df[(df['annual_inc'] >= 15000) & (df['annual_inc'] <= 300000)].copy()
df_filtered['outcome'] = df_filtered['default_flag'].map({1: 'Defaulted', 0: 'Paid'})

sns.kdeplot(
    data=df_filtered,
    x='annual_inc',
    hue='outcome',
    fill=True,
    alpha=0.4,
    palette={'Defaulted': '#EF5350', 'Paid': '#4FC3F7'},
    linewidth=2,
    ax=ax
)

ax.set_title('Income Distribution by Loan Outcome', fontsize=15, pad=15)
ax.set_xlabel('Annual Income ($)', fontsize=12)
ax.yaxis.set_visible(False)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('v2_income_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Visualization 3 - Default Rate by DTI Group

DTI (Debt-to-Income Ratio) measures how much of a borrower's monthly income was already committed to debt payments before taking this loan. A DTI of 30 means 30% of income was already spoken for.

Borrowers with DTI above 30 default at nearly double the rate of those with DTI below 10. This is the most direct evidence of over-indebtedness in the dataset, and it is consistent with the BCCR hypothesis that accumulated debt - not just income levels - drives delinquency.

In [ ]:
# Data sourced from SQL Analysis 4 (credit_risk_queries.sql)
# Query: DEFAULT RATE BY DTI GROUP

plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(10, 6))

dti_data = {
    'DTI Group': ['Low (0-10)', 'Moderate (10-20)', 'High (20-30)', 'Very High (30+)'],
    'Default Rate': [16.11, 18.99, 24.49, 30.47]
}

dti_df = pd.DataFrame(dti_data)

bars = ax.bar(
    dti_df['DTI Group'],
    dti_df['Default Rate'],
    color=['#4FC3F7', '#81D4FA', '#EF9A9A', '#EF5350'],
    width=0.6,
    edgecolor='none'
)

for bar, val in zip(bars, dti_df['Default Rate']):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        f'{val}%',
        ha='center', va='bottom',
        fontsize=12, color='white'
    )

ax.set_title('Default Rate by Debt-to-Income Ratio Group', fontsize=15, pad=15)
ax.set_xlabel('DTI Group', fontsize=12)
ax.set_ylabel('Default Rate (%)', fontsize=12)
ax.set_ylim(0, 36)
ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
plt.savefig('v3_dti_default_rate.png', dpi=150, bbox_inches='tight')
plt.show()